# JailbreakBench (Mistral-7B) + LMSYS Benign Evaluation

Two independent evaluation workflows in this notebook:

**Part A — JailbreakBench / Mistral-7B** (port 8000)  
Runs the 100 JBB-Behaviors goals against `mistralai/Mistral-7B-Instruct-v0.3`.

**Part B — LMSYS Benign Prompts** (port 8001)  
Samples real-world benign prompts from `lmsys/lmsys-chat-1m` and runs them against the same endpoint.  
Requires HuggingFace login + licence acceptance at https://huggingface.co/datasets/lmsys/lmsys-chat-1m

Both loops are **synchronous** — each request blocks until the previous one completes.  
No artificial sleep is used; natural round-trip time controls pacing.

---
## 1. Install dependencies

In [ ]:
!pip install datasets huggingface_hub pandas matplotlib ipywidgets

---
# PART A — JailbreakBench / Mistral-7B
## 2. Configuration

In [ ]:
import requests
import pandas as pd
import json
import time
from datetime import datetime

JBB_ENDPOINT  = "http://192.168.2.229:8000/v1/completions"
JBB_MODEL     = "mistralai/Mistral-7B-Instruct-v0.3"
TIMEOUT_S     = 60

## 3. Verify endpoint

In [ ]:
try:
    health = requests.get("http://192.168.2.229:8000/health", timeout=5)
    print(f"✓ vLLM health: {health.status_code}")
    models = requests.get("http://192.168.2.229:8000/v1/models", timeout=5).json()
    print(f"✓ Loaded models: {[m['id'] for m in models.get('data', [])]}")
except Exception as e:
    print(f"✗ Cannot reach vLLM: {e}")

## 4. Load JailbreakBench behaviors

In [ ]:
from datasets import load_dataset

print("Loading JailbreakBench behaviors...")
dataset = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors")

split_name  = list(dataset.keys())[0]
behaviors_df = pd.DataFrame(dataset[split_name])

print(f"✓ Loaded {len(behaviors_df)} behaviors")
print(f"Columns : {behaviors_df.columns.tolist()}")
behaviors_df.head(3)

## 5. Run all 100 JBB-Behaviors against Mistral-7B

In [ ]:
# Column names vary slightly across JBB dataset versions — handle both
goal_col     = 'Goal'     if 'Goal'     in behaviors_df.columns else 'goal'
category_col = 'Category' if 'Category' in behaviors_df.columns else 'category'

jbb_results = []
jbb_failed  = []

print(f"Running {len(behaviors_df)} behaviors...\n")

for idx, row in behaviors_df.iterrows():
    goal     = row[goal_col]
    category = row[category_col]

    payload = {
        "model"      : JBB_MODEL,
        "prompt"     : goal,
        "max_tokens" : 100,
        "temperature": 0.7,
    }

    # Synchronous — next request waits for this one to complete
    start = time.time()
    try:
        r       = requests.post(JBB_ENDPOINT, json=payload, timeout=TIMEOUT_S)
        r.raise_for_status()
        data    = r.json()
        content = data["choices"][0]["text"]
        tokens  = data["usage"]["completion_tokens"]
    except Exception as e:
        jbb_failed.append({"idx": idx, "goal": goal[:80], "error": str(e)})
        print(f"✗ Error on idx {idx}: {e}")
        continue
    latency = time.time() - start

    jbb_results.append({
        "idx"             : idx,
        "timestamp"       : datetime.now().isoformat(),
        "goal"            : goal,
        "category"        : category,
        "response"        : content,
        "response_tokens" : tokens,
        "latency_seconds" : round(latency, 4),
    })

    if (idx + 1) % 10 == 0:
        print(f"  {idx + 1}/{len(behaviors_df)} — latency: {latency:.2f}s")

jbb_df = pd.DataFrame(jbb_results)
print(f"\n✓ Completed : {len(jbb_df)} queries")
print(f"✗ Failed    : {len(jbb_failed)} queries")
print(f"Mean latency: {jbb_df['latency_seconds'].mean():.2f}s")

jbb_df.to_json("jailbreak_results.jsonl", orient="records", lines=True)
jbb_df.to_csv("jailbreak_results.csv", index=False)
print("\nSaved: jailbreak_results.jsonl / .csv")

# Summary by category
print("\n=== Summary by category ===")
print(jbb_df.groupby("category").agg(
    count=("latency_seconds", "count"),
    avg_latency=("latency_seconds", "mean"),
    avg_tokens=("response_tokens", "mean"),
    tokens_per_sec=("response_tokens", lambda x: (x / jbb_df.loc[x.index, "latency_seconds"]).mean())
).round(2))

if jbb_failed:
    print(f"\nFailed rows:")
    for f in jbb_failed:
        print(f"  idx {f['idx']}: {f['error'][:80]}")

---
# PART B — LMSYS Benign Prompts
## 6. HuggingFace login

> **Token security**: never paste a token directly into a notebook cell.  
> Run `huggingface-cli login` in your terminal once, or set the `HUGGINGFACE_TOKEN`  
> environment variable, and call `login()` with no arguments here.

In [ ]:
from huggingface_hub import login

# Uses cached token from `huggingface-cli login` or HUGGINGFACE_TOKEN env var
# Never hardcode a token in this cell
login()

## 7. Load & filter LMSYS benign prompts

In [ ]:
import matplotlib.pyplot as plt

N_BENIGN = 8000  # target sample size

print("Loading lmsys-chat-1m (this may take a few minutes)...")
lmsys_raw = load_dataset("lmsys/lmsys-chat-1m", split="train")

# Extract first user turn only
def extract_first_user_turn(example):
    for msg in example["conversation"]:
        if msg["role"] == "user":
            return {"prompt": msg["content"], "language": example.get("language", "unknown")}
    return {"prompt": None, "language": None}

lmsys_mapped = lmsys_raw.map(extract_first_user_turn)
df_full      = lmsys_mapped.to_pandas()

# Filter: English, not flagged by OpenAI moderation, reasonable length
df_filtered = df_full[
    (df_full["language"] == "English") &
    (df_full["openai_moderation"].apply(
        lambda x: not x["flagged"] if isinstance(x, dict) else True
    )) &
    (df_full["prompt"].str.len() > 30) &
    (df_full["prompt"].str.len() < 2000) &
    (df_full["prompt"].notna())
]

benign_df = df_filtered.sample(n=N_BENIGN, random_state=42)[["prompt"]].reset_index(drop=True)
benign_df["label"] = 0

print(f"✓ Pool after filtering : {len(df_filtered):,} rows")
print(f"✓ Sampled              : {len(benign_df):,} rows")
print(f"Null prompts           : {benign_df['prompt'].isna().sum()}")
print(f"Duplicate prompts      : {benign_df['prompt'].duplicated().sum()}")

benign_df["prompt_len"] = benign_df["prompt"].str.len()
print(f"\nPrompt length stats:")
print(benign_df["prompt_len"].describe().round(1))

benign_df["prompt_len"].hist(bins=50, figsize=(10, 4))
plt.title("LMSYS Benign Prompt Length Distribution")
plt.xlabel("Character length")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

benign_df.drop(columns=["prompt_len"]).to_csv("benign_prompts.csv", index=False)
print("\nSaved: benign_prompts.csv")

## 8. Smoke test — verify endpoint before full run

In [ ]:
LMSYS_ENDPOINT = "http://192.168.2.229:8001/v1/completions"
LMSYS_TIMEOUT  = 60

# Check available models on port 8001
print(requests.get("http://192.168.2.229:8001/v1/models", timeout=5).json())

# Single-prompt test
test_payload = {
    "prompt"     : benign_df.iloc[0]["prompt"],
    "max_tokens" : 256,
    "temperature": 0.0,
}
test_resp = requests.post(LMSYS_ENDPOINT, json=test_payload, timeout=LMSYS_TIMEOUT)
print(json.dumps(test_resp.json(), indent=2))

## 9. Run 8,000 LMSYS benign prompts

In [ ]:
lmsys_results = []
lmsys_failed  = []

print(f"Running {len(benign_df)} benign prompts...\n")

for i, row in benign_df.iterrows():
    prompt = row["prompt"]

    payload = {
        "prompt"     : prompt,
        "max_tokens" : 256,
        "temperature": 0.0,
    }

    # Synchronous — next request waits for this one to complete
    start = time.time()
    try:
        r       = requests.post(LMSYS_ENDPOINT, json=payload, timeout=LMSYS_TIMEOUT)
        r.raise_for_status()
        data    = r.json()
        content = data["choices"][0]["text"]
        tokens  = data["usage"]["completion_tokens"]
    except Exception as e:
        lmsys_failed.append({"idx": i, "prompt": prompt[:80], "error": str(e)})
        print(f"✗ Error on idx {i}: {e}")
        continue
    latency = time.time() - start

    lmsys_results.append({
        "idx"             : i,
        "timestamp"       : datetime.now().isoformat(),
        "prompt"          : prompt,
        "response"        : content,
        "response_tokens" : tokens,
        "latency_seconds" : round(latency, 4),
        "abuse_label"     : 0,   # 0 = benign
    })

    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(benign_df)} — latency: {latency:.2f}s")

lmsys_df = pd.DataFrame(lmsys_results)
print(f"\n✓ Completed : {len(lmsys_df)} queries")
print(f"✗ Failed    : {len(lmsys_failed)} queries")
print(f"Mean latency: {lmsys_df['latency_seconds'].mean():.2f}s")
print(f"Error count : {len(lmsys_failed)}")

lmsys_df.to_csv("lmsys_benign_8k.csv", index=False)
lmsys_df.to_json("lmsys_benign_8k.jsonl", orient="records", lines=True)
print("\nSaved: lmsys_benign_8k.csv / .jsonl")

if lmsys_failed:
    print(f"\nFailed rows:")
    for f in lmsys_failed:
        print(f"  idx {f['idx']}: {f['error'][:80]}")